# Reconstruction des métadonnées de GSE149689

Reconstruction des métadonnées des 20 échantillons à partir des suffixes des barcodes, sans QC ni analyse transcriptomique.

Sources : [GEO GSE149689](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE149689) et [article associé](https://pmc.ncbi.nlm.nih.gov/articles/PMC7402635/). Les statuts COVID-19 proviennent des fiches GSM de GEO. L'article décrit les cinq échantillons influenza comme des cas sévères. La sévérité des témoins sains est laissée manquante.

In [1]:
import pandas as pd
import scanpy as sc

## Chargement de l'objet brut

In [2]:
adata = sc.read_h5ad("../data/processed/GSE149689_raw.h5ad")

## Extraction de `sample_id` depuis le suffixe des barcodes

In [3]:
barcode_suffix = adata.obs_names.to_series(index=adata.obs_names).str.extract(
    r"-(\d+)$", expand=False
)
if barcode_suffix.isna().any():
    raise ValueError("Au moins un barcode ne possède pas de suffixe numérique.")

adata.obs["sample_id"] = "Sample" + barcode_suffix.astype(int).astype(str)

## Table officielle des métadonnées échantillon

In [4]:
sample_metadata = pd.DataFrame(
    [
        ("Sample1", "SW103", "COVID-19", "severe"),
        ("Sample2", "SW104", "COVID-19", "mild"),
        ("Sample3", "SW105", "influenza", "severe"),
        ("Sample4", "SW106", "influenza", "severe"),
        ("Sample5", "SW107", "healthy_control", pd.NA),
        ("Sample6", "SW108", "influenza", "severe"),
        ("Sample7", "SW109", "influenza", "severe"),
        ("Sample8", "SW110", "influenza", "severe"),
        ("Sample9", "SW111", "COVID-19", "severe"),
        ("Sample10", "SW112", "COVID-19", "severe"),
        ("Sample11", "SW113", "COVID-19", "mild"),
        ("Sample12", "SW114", "COVID-19", "mild"),
        ("Sample13", "SW115", "healthy_control", pd.NA),
        ("Sample14", "SW116", "healthy_control", pd.NA),
        ("Sample15", "SW117", "COVID-19", "severe"),
        ("Sample16", "SW118", "COVID-19", "severe"),
        ("Sample17", "SW119", "COVID-19", "severe"),
        ("Sample18", "SW120", "COVID-19", "mild"),
        ("Sample19", "SW121", "healthy_control", pd.NA),
        ("Sample20", "SW122", "COVID-19", "asymptomatic"),
    ],
    columns=["sample_id", "subject_id", "disease", "severity"],
)

## Fusion avec `adata.obs`

In [5]:
adata.obs = adata.obs.join(
    sample_metadata.set_index("sample_id"),
    on="sample_id",
    validate="many_to_one",
)

## Aperçu et distributions demandées

In [6]:
display(adata.obs.head())
display(adata.obs["sample_id"].value_counts().sort_index())
display(adata.obs["disease"].value_counts())
display(adata.obs["severity"].value_counts(dropna=False))

,sample_id,subject_id,disease,severity
AAACCCAAGGGCAATC-1,Sample1,SW103,COVID-19,severe
AAACCCAAGGTGCCAA-1,Sample1,SW103,COVID-19,severe
AAACCCACAAGAATGT-1,Sample1,SW103,COVID-19,severe
AAACCCACAGCTGAAG-1,Sample1,SW103,COVID-19,severe
AAACCCACATATCGGT-1,Sample1,SW103,COVID-19,severe


sample_id
Sample1     6455
Sample10    1167
Sample11    6398
Sample12    6283
Sample13    6156
Sample14    6574
Sample15    2349
Sample16    4371
Sample17    1755
Sample18    3984
Sample19    5542
Sample2     7731
Sample20    4868
Sample3     5851
Sample4     1724
Sample5     6426
Sample6     3516
Sample7     1455
Sample8     2001
Sample9      538
Name: count, dtype: int64

disease
COVID-19           45899
healthy_control    24698
influenza          14547
Name: count, dtype: int64

severity
severe          31182
<NA>            24698
mild            24396
asymptomatic     4868
Name: count, dtype: int64

## Table résumée et contrôles

In [7]:
n_cells = (
    adata.obs.groupby("sample_id", observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)
sample_summary = sample_metadata.merge(
    n_cells, on="sample_id", how="left", validate="one_to_one"
)

expected_suffix = adata.obs["sample_id"].str.removeprefix("Sample")
checks = {
    "exactly_20_sample_ids": adata.obs["sample_id"].nunique() == 20,
    "no_missing_sample_id": not adata.obs["sample_id"].isna().any(),
    "no_missing_disease": not adata.obs["disease"].isna().any(),
    "suffix_matches_sample_id": bool((barcode_suffix == expected_suffix).all()),
    "one_row_per_sample": len(sample_summary) == 20 and sample_summary["sample_id"].is_unique,
    "all_cells_counted": int(sample_summary["n_cells"].sum()) == adata.n_obs,
}
if not all(checks.values()):
    raise AssertionError({name: passed for name, passed in checks.items() if not passed})

## Résultat final

In [8]:
display(sample_summary)
print("Contrôles :", ", ".join(f"{name}=OK" for name in checks))
print("Sévérité manquante uniquement pour les 4 témoins sains :", int(sample_summary["severity"].isna().sum()) == 4)

,sample_id,subject_id,disease,severity,n_cells
0,Sample1,SW103,COVID-19,severe,6455
1,Sample2,SW104,COVID-19,mild,7731
2,Sample3,SW105,influenza,severe,5851
3,Sample4,SW106,influenza,severe,1724
4,Sample5,SW107,healthy_control,<NA>,6426
5,Sample6,SW108,influenza,severe,3516
6,Sample7,SW109,influenza,severe,1455
7,Sample8,SW110,influenza,severe,2001
8,Sample9,SW111,COVID-19,severe,538
9,Sample10,SW112,COVID-19,severe,1167


Contrôles : exactly_20_sample_ids=OK, no_missing_sample_id=OK, no_missing_disease=OK, suffix_matches_sample_id=OK, one_row_per_sample=OK, all_cells_counted=OK
Sévérité manquante uniquement pour les 4 témoins sains : True
